# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets (@id) in the dataset
record_sets = list(dataset.record_sets)
print("Record sets in the dataset:")
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs['name']}")

# For each record set, list available fields with @id
print("\nFields in record sets:")
for rs in record_sets:
    print(f"\nRecord Set: {rs['name']} (@id: {rs['@id']})")
    fields = rs.get("field", [])
    if not isinstance(fields, list):
        fields = [fields]
    for f in fields:
        field_obj = dataset.get_field(f['@id']) if isinstance(f, dict) and '@id' in f else None
        if field_obj:
            print(f"   - Field: {field_obj.name}, @id: {field_obj['@id']}, dataType: {getattr(field_obj, 'dataType', 'N/A')}")
        else:
            print(f"   - Field @id: {f['@id']}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose a record set to extract (update these variables based on printed overview above)
# Example: protein_record_set_id = 'https://api.app.sen.science/frontiers/7154140/7cd93b38-588b-4e83-8359-0b1397e7b9c2'
protein_record_set_id = None
for rs in dataset.record_sets:
    if 'protein' in rs['name'].lower() or 'abundance' in rs['name'].lower():
        protein_record_set_id = rs['@id']
        break
if protein_record_set_id is None:
    protein_record_set_id = dataset.record_sets[0]['@id']  # fallback to first dataset
print(f"Selected record set: {protein_record_set_id}")

# If there are multiple record sets, extract all into dataframes
dataframes = {}
for rs in dataset.record_sets:
    rs_id = rs['@id']
    print(f"Loading records for: {rs['name']} (@id: {rs_id})")
    try:
        records = list(dataset.records(record_set=rs_id))
        if len(records):
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"  - Shape: {df.shape}")
            print(f"  - Columns: {df.columns.tolist()}")
        else:
            print("  - No records available.")
    except Exception as e:
        print(f"  - Failed to load: {e}")

# Print columns of the main table
if protein_record_set_id in dataframes:
    print(f"\nColumns in {protein_record_set_id}:")
    print(dataframes[protein_record_set_id].columns.tolist())
    display(dataframes[protein_record_set_id].head())
else:
    print(f"No data for primary record set id: {protein_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
# Select the numeric field to analyze (change field id as appropriate)
df = dataframes[protein_record_set_id]

# Choose a numeric field that appears to be protein abundance or MW, or fallback to first float column
candidate_numeric_fields = [c for c in df.columns if any(word in c.lower() for word in ["abundance", "mw", "molecular_weight", "coverage", "score", "count"]) and pd.api.types.is_numeric_dtype(df[c])]
if not candidate_numeric_fields:
    candidate_numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
    print(f"Selected numeric field: {numeric_field}")
else:
    print("No obvious numeric field in table, cannot proceed with EDA.")

# Filter for high values
if candidate_numeric_fields:
    threshold = np.nanmean(df[numeric_field])
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field (@id)
    candidate_cats = [c for c in df.columns if any(name in c.lower() for name in ["description", "group", "sample", "accession", "class"])]
    group_field = candidate_cats[0] if candidate_cats else df.columns[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if candidate_numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=40, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If a grouping field exists, top N groups
    top_groups = grouped_df.sort_values(by=numeric_field, ascending=False).head(10)
    plt.figure(figsize=(10, 4))
    sns.barplot(x=group_field, y=numeric_field, data=top_groups)
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.xticks(rotation=60)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to explore a Croissant-based dataset for protein analysis using `mlcroissant`.
- We loaded structured data, extracted tables by record set `@id`, explored field types, and visualized quantitative properties.
- You can repeat the EDA steps for other record sets or fields as needed for deeper analysis of protein, peptide, and sample attributes.